# PPThinning

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `full`
- Workflow family: `network`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/PPThinning.md`


Notebook source link: [PPThinning.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/PPThinning.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "PPThinning"
FAMILY = "network"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"PPThinning: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"PPThinning: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"PPThinning: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"PPThinning: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "close all;",
    "delta = 0.001;",
    "Tmax = 100;",
    "time = 0:delta:Tmax;",
    "f=.1;",
    "lambdaData = 10*sin(2*pi*f*time)+10; %lambda >=0",
    "lambda = Covariate(time,lambdaData, '\\Lambda(t)','time','s','Hz',{'\\lambda_{1}'},{{' ''b'', ''LineWidth'' ,2'}});",
    "lambdaBound = max(lambda);",
    "N=lambdaBound*(1.5*Tmax);   %Expected number of arrivals in interval 1.5*Tmax",
    "u = rand(1,N);              %N samples uniform(0,1)",
    "w = -log(u)./(lambdaBound); %N samples exponential rate lambdaBound (ISIs)",
    "tSpikes = cumsum(w);        %Spiketimes;",
    "tSpikes = tSpikes(tSpikes<=Tmax);%Spiketimes within Tmax",
    "lambdaRatio = lambda.getValueAt(tSpikes)./lambdaBound;",
    "u2 = rand(length(lambdaRatio),1);",
    "tSpikesThin  = tSpikes(lambdaRatio>=u2);",
    "figure(1);",
    "n1 = nspikeTrain(tSpikes);",
    "n2 = nspikeTrain(tSpikesThin);",
    "subplot(2,2,1); n1.plot; plot(tSpikes,ones(size(tSpikes)),'.');",
    "v=axis; axis([0 Tmax/4 v(3) v(4)]);",
    "subplot(2,2,2); n1.plotISIHistogram;",
    "subplot(2,2,3); n2.plot; plot(tSpikes,ones(size(tSpikes)),'.');",
    "v=axis; axis([0 Tmax/4 v(3) v(4)]);",
    "subplot(2,2,4); n2.plotISIHistogram;",
    "figure(2);",
    "n2.plot;",
    "scaledProb = lambda*(1./lambdaBound);",
    "scaledProb.plot;",
    "v=axis;",
    "axis([0 Tmax/4 v(3) v(4)]);",
    "numRealizations = 20;",
    "spikeColl = CIF.simulateCIFByThinningFromLambda(lambda,numRealizations);",
    "figure(3);",
    "spikeColl.plot;",
    "lambda.plot;",
    "v=axis;",
    "axis([0 Tmax/4 v(3) v(4)]);",
    "parity = struct();",
    "parity.num_realizations = numRealizations;"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)
print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for PPThinning.")


In [ ]:
# PPThinning: thinning-based spike simulation from a known CIF.
delta = 0.001
Tmax = 100.0
time = np.arange(0.0, Tmax + delta, delta)
f = 0.1
lambda_data = 10.0 * np.sin(2.0 * np.pi * f * time) + 10.0
lambda_bound = float(np.max(lambda_data))

# Generate candidate spikes from homogeneous Poisson process at lambda_bound.
N = int(np.ceil(lambda_bound * (1.5 * Tmax)))
u = rng.random(N)
w = -np.log(np.clip(u, 1e-12, 1.0)) / lambda_bound
t_spikes = np.cumsum(w)
t_spikes = t_spikes[t_spikes <= Tmax]

idx = np.clip(np.rint(t_spikes / delta).astype(int), 0, time.size - 1)
lambda_ratio = lambda_data[idx] / lambda_bound
u2 = rng.random(lambda_ratio.size)
t_spikes_thin = t_spikes[lambda_ratio >= u2]

# MATLAB Figure 1: candidate-vs-thinned rasters and ISI histograms.
fig1, axes = plt.subplots(2, 2, figsize=(10, 6.8))
axes[0, 0].vlines(t_spikes, 0.0, 1.0, color="k", linewidth=0.5)
axes[0, 0].set_xlim(0.0, Tmax / 4.0)
axes[0, 0].set_yticks([])
axes[0, 0].set_title("Constant-rate process")

isi_raw = np.diff(t_spikes)
axes[0, 1].hist(isi_raw, bins=60, color="0.35")
axes[0, 1].set_title("ISI histogram (constant rate)")

axes[1, 0].vlines(t_spikes_thin, 0.0, 1.0, color="k", linewidth=0.5)
axes[1, 0].set_xlim(0.0, Tmax / 4.0)
axes[1, 0].set_yticks([])
axes[1, 0].set_title("Thinned process")

isi_thin = np.diff(t_spikes_thin) if t_spikes_thin.size > 1 else np.array([0.0])
axes[1, 1].hist(isi_thin, bins=60, color="0.35")
axes[1, 1].set_title("ISI histogram (thinned)")
for ax in axes.ravel():
    ax.set_xlabel("time [s]")
plt.tight_layout()
plt.show()

# MATLAB Figure 2: thinned spikes + scaled intensity.
fig2, ax2 = plt.subplots(1, 1, figsize=(9, 4.2))
ax2.vlines(t_spikes_thin, 0.0, 1.0, color="k", linewidth=0.5, label="thinned spikes")
ax2.plot(time, lambda_data / lambda_bound, "b", linewidth=1.2, label="lambda/lambda_max")
ax2.set_xlim(0.0, Tmax / 4.0)
ax2.set_ylim(0.0, 1.05)
ax2.set_xlabel("time [s]")
ax2.set_title("Thinned raster and acceptance probability")
ax2.legend(loc="upper right")
plt.tight_layout()
plt.show()

# MATLAB Figure 3/4 style: multiple realizations against CIF.
n_real = 20
raster = []
for _ in range(n_real):
    keep = t_spikes[rng.random(t_spikes.size) <= lambda_ratio]
    raster.append(keep)

fig3, (ax31, ax32) = plt.subplots(2, 1, figsize=(9, 6.8), sharex=True)
for i, spk in enumerate(raster):
    ax31.vlines(spk, i + 0.6, i + 1.4, color="k", linewidth=0.4)
ax31.set_xlim(0.0, Tmax / 4.0)
ax31.set_ylabel("realization")
ax31.set_title("Thinning-generated sample paths")

ax32.plot(time, lambda_data, "b", linewidth=1.2)
ax32.set_xlim(0.0, Tmax / 4.0)
ax32.set_xlabel("time [s]")
ax32.set_ylabel("Hz")
ax32.set_title("Conditional intensity function")
plt.tight_layout()
plt.show()

fig4, ax4 = plt.subplots(1, 1, figsize=(9, 3.8))
bins = np.arange(0.0, Tmax + 0.25, 0.25)
stacked = []
for spk in raster:
    hist, _ = np.histogram(spk, bins=bins)
    stacked.append(hist)
stacked = np.asarray(stacked, dtype=float)
ax4.plot(0.5 * (bins[:-1] + bins[1:]), np.mean(stacked, axis=0) / 0.25, "k", linewidth=1.3, label="mean rate")
ax4.plot(time, lambda_data, "b--", linewidth=1.0, label="true lambda(t)")
ax4.set_xlim(0.0, Tmax / 4.0)
ax4.set_xlabel("time [s]")
ax4.set_ylabel("Hz")
ax4.set_title("Empirical mean rate vs. CIF")
ax4.legend(loc="upper right")
plt.tight_layout()
plt.show()

accept_ratio = float(t_spikes_thin.size / max(t_spikes.size, 1))
print("accepted", t_spikes_thin.size, "candidates", t_spikes.size, "ratio", accept_ratio)
assert t_spikes_thin.size > 20
assert 0.0 < accept_ratio < 1.0

CHECKPOINT_METRICS = {
    "accepted_spike_count": float(t_spikes_thin.size),
    "accept_ratio": float(accept_ratio),
}
CHECKPOINT_LIMITS = {
    "accepted_spike_count": (20.0, 50000.0),
    "accept_ratio": (0.01, 0.99),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
